In [7]:
import sys
import os

import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data_loader import load_raw_data
from src.preprocessing import split_data, impute_missing_values, scale_features, COLS_TO_IMPUTE

In [8]:
df = load_raw_data()
print(f"Dimensions du dataset : {df.shape}")
df.head()

Dimensions du dataset : (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 1. Séparation Entraînement / Test (Train/Test Split)

Avant toute transformation, on divise les données en deux ensembles :
- **Train (80%)** : servira à entraîner les modèles *et* à calculer les statistiques de nettoyage (médiane, σ du scaler...)
- **Test (20%)** : servira uniquement à évaluer les modèles, comme s'il s'agissait de nouvelles données inconnues

> **Pourquoi faire ça en premier ?**
> On réalise cette étape en premier pour éviter le Data Leakage.
> Le Data Leakage, c'est quand des informations du **test** "contaminent" le nettoyage des données.
>
> Par exemple si on calcule la médiane de l'Insuline sur *tout* le dataset avant de splitter,
> cette médiane est influencée par les données de test. Le modèle tire profit d'une information
> qu'il ne devrait pas avoir → ses performances en production seront **surestimées**.
>
> **La règle d'or :** tout ce qu'on "apprend" depuis les données (médiane, σ...) doit être
> calculé **uniquement sur le train**, puis *appliqué* sur le test.

On utilise `stratify=y` pour garantir que la proportion de diabétiques (Outcome=1) est la même dans les deux ensembles — comme vous l'avez observé : **34.85% train ≈ 35.06% test**.

In [9]:
X_train, X_test, y_train, y_test = split_data(df)

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"Proportion Outcome=1 dans train : {y_train.mean():.2%}")
print(f"Proportion Outcome=1 dans test : {y_test.mean():.2%}")

X_train : (614, 8) | X_test : (154, 8)
Proportion Outcome=1 dans train : 34.85%
Proportion Outcome=1 dans test : 35.06%


## 2. Imputation des valeurs manquantes

Lors de l'EDA, on a identifié des zéros médicalement impossibles dans 5 colonnes :
Glucose, BloodPressure, SkinThickness, Insulin, BMI.

On les remplace par la **médiane** (plutôt que la moyenne) car :
- La médiane est robuste aux valeurs extrêmes (outliers) observées dans l'EDA
- La médiane est calculée **uniquement sur X_train**, puis appliquée au test → pas de data leakage

In [10]:
X_train_clean, X_test_clean = impute_missing_values(X_train, X_test)

cols = COLS_TO_IMPUTE

print("Zéros restants après imputation :")
for c in cols:
    zeros_avant = (X_train[c] == 0).sum()
    zeros_apres = (X_train_clean[c] == 0).sum()
    print(f" {c:25s} : {zeros_avant} zéros → {zeros_apres} zéros")

Zéros restants après imputation :
 Glucose                   : 4 zéros → 0 zéros
 BloodPressure             : 23 zéros → 0 zéros
 SkinThickness             : 175 zéros → 0 zéros
 Insulin                   : 290 zéros → 0 zéros
 BMI                       : 9 zéros → 0 zéros


## 3. Mise à l'échelle (Feature Scaling)

Les variables ont des plages très différentes (ex: Insuline jusqu'à 800, DiabetesPedigreeFunction entre 0 et 2.4).
Les modèles comme la Régression Logistique sont sensibles à cette disparité.

On utilise la **Standardisation (StandardScaler)** qui transforme chaque variable pour avoir :
- Moyenne = 0 et Écart-type = 1
- Formule : `x_scaled = (x - μ) / σ`

Comme pour l'imputation, μ et σ sont calculés **uniquement sur X_train**.

In [12]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train_clean, X_test_clean)

print("Moyennes par colonne (doit être ≈ 0) :")
print(X_train_scaled.mean(axis=0).round(4))

print("\nÉcarts-types par colonne (doit être ≈ 1) :")
print(X_train_scaled.std(axis=0, ddof=0).round(4))

Moyennes par colonne (doit être ≈ 0) :
Pregnancies                -0.0
Glucose                    -0.0
BloodPressure               0.0
SkinThickness              -0.0
Insulin                    -0.0
BMI                         0.0
DiabetesPedigreeFunction   -0.0
Age                        -0.0
dtype: float64

Écarts-types par colonne (doit être ≈ 1) :
Pregnancies                 1.0
Glucose                     1.0
BloodPressure               1.0
SkinThickness               1.0
Insulin                     1.0
BMI                         1.0
DiabetesPedigreeFunction    1.0
Age                         1.0
dtype: float64


In [13]:
X_train_scaled.to_csv("../data/processed/X_train_clean.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test_clean.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)